# T03b — Đề xuất mask hiện vật (Grounding DINO + SAM 2)

Notebook chạy trên **keyframe thật** của ba camera. Kết quả chỉ là đề xuất `needs_review`; không tự động thay ROI hoặc bật rule.

In [ ]:
import os, subprocess, sys, torch
assert torch.cuda.is_available(), 'Hãy chọn Runtime > Change runtime type > T4 GPU rồi chạy lại.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!pip -q install "transformers>=4.56.0" accelerate safetensors opencv-python-headless
!pip -q install git+https://github.com/facebookresearch/sam2.git

Tải file `artifact_mask_payload.zip` ở máy lên khi hộp chọn file xuất hiện.

In [ ]:
from google.colab import files
uploaded = files.upload()
assert 'artifact_mask_payload.zip' in uploaded, 'Cần tải đúng file artifact_mask_payload.zip'
!rm -rf /content/artifact_mask_job /content/artifact_mask_results.zip
!mkdir -p /content/artifact_mask_job
!unzip -q /content/artifact_mask_payload.zip -d /content/artifact_mask_job
%cd /content/artifact_mask_job
!find . -maxdepth 3 -type f | sort

In [ ]:
!python -u -m src.artifacts.generate_mask_proposals --config configs/artifact_detection.json --output-dir outputs/artifact_mask_proposals --device cuda

In [ ]:
!python -m src.artifacts.validate_mask_proposals outputs/artifact_mask_proposals --expected-count 8
import json
from pathlib import Path
rows = [json.loads(x) for x in Path('outputs/artifact_mask_proposals/mask_proposals.jsonl').read_text().splitlines()]
for r in rows:
    print(f"cam{r['camera_id']} {r['artifact_id']}: {r['proposal_method']}, DINO={r['grounding_dino_score']}, flags={r['quality_flags']}")

In [ ]:
!cd outputs && zip -qr /content/artifact_mask_results.zip artifact_mask_proposals
from google.colab import files
files.download('/content/artifact_mask_results.zip')

Sau khi tải xong, gửi lại `artifact_mask_results.zip`. Mình sẽ dựng trang duyệt overlay/mask, loại mask sai và chỉ tích hợp những mask được xác nhận.